## PHASE 6: MODEL BUILDING & CLASSIFICATION
**Duration:** 2-3 weeks | **Deliverable:** trained_models/, model_predictions.csv

### Step 6.1: Train Multiple Classifiers

```python
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Model 1: Random Forest (good for mixed feature types)
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',  # Penalize errors on minority class
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_balanced, y_train_balanced)

# Model 2: XGBoost (often superior)
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    scale_pos_weight=19,  # Ratio of negatives to positives
    random_state=42
)
xgb_model.fit(X_train_balanced, y_train_balanced)

# Model 3: Logistic Regression (baseline)
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_scaled, y_train_balanced)

# Store all models
models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'Logistic Regression': lr_model
}
```

### Step 6.2: Make Predictions

```python
predictions = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
    
    predictions[name] = {
        'predictions': preds,
        'probabilities': probs
    }
```

---

## PHASE 7: EVALUATION & METRICS
**Duration:** 1 week | **Deliverable:** evaluation_report.xlsx, model_comparison.csv

### Step 7.1: Compute Comprehensive Metrics

```python
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    f1_score,
    roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt

for name, pred_dict in predictions.items():
    y_pred = pred_dict['predictions']
    y_pred_proba = pred_dict['probabilities']
    
    print(f"\n{'='*60}")
    print(f"MODEL: {name}")
    print(f"{'='*60}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"\nConfusion Matrix:")
    print(f"  True Negatives:  {tn:5d} (correctly identified legitimate)")
    print(f"  False Positives: {fp:5d} (legitimate flagged as fraud)")
    print(f"  False Negatives: {fn:5d} (fraud missed) ⚠️ CRITICAL")
    print(f"  True Positives:  {tp:5d} (correctly identified fraud)")
    
    # Key Metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"\nKey Metrics:")
    print(f"  Precision: {precision:.4f} (of detected fraud, how many are real)")
    print(f"  Recall:    {recall:.4f} (of actual fraud, how many did we catch) ⚠️ PRIORITY")
    print(f"  F1-Score:  {f1:.4f} (balance between precision & recall)")
    print(f"  AUC-ROC:   {auc:.4f}")
    
    # Classification Report
    print(f"\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))
```

### Step 7.2: Feature Importance Analysis

```python
# For Random Forest
feature_importance = pd.DataFrame({
    'feature': all_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features (Random Forest):")
print(feature_importance.head(20).to_string())

# Highlight: are mined red flags in the top features?
red_flag_importance = feature_importance[
    feature_importance['feature'].isin(red_flag_features)
].head(10)
print("\nRed Flag Feature Importance:")
print(red_flag_importance.to_string())
```

### Step 7.3: Impact Analysis - Did Mining Help?

**Compare models with and without mined features:**

```python
# Train Random Forest WITHOUT red flag features
features_without_mining = [f for f in all_features if f not in red_flag_features]
X_train_no_mining = X_train_balanced[features_without_mining]
X_test_no_mining = X_test[features_without_mining]

rf_no_mining = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced',
    random_state=42
)
rf_no_mining.fit(X_train_no_mining, y_train_balanced)

# Compare
from sklearn.metrics import f1_score
f1_with_mining = f1_score(y_test, rf_model.predict(X_test))
f1_without_mining = f1_score(y_test, rf_no_mining.predict(X_test_no_mining))

print(f"\nMining Impact on Model Performance:")
print(f"  F1-Score WITH mined features:    {f1_with_mining:.4f}")
print(f"  F1-Score WITHOUT mined features: {f1_without_mining:.4f}")
print(f"  Improvement: {(f1_with_mining - f1_without_mining):.4f} ({(f1_with_mining/f1_without_mining - 1)*100:.1f}%)")
```

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    f1_score,
)

try:
    from xgboost import XGBClassifier, XGBRegressor
    xgb_available = True
except ImportError:
    xgb_available = False
    from sklearn.ensemble import HistGradientBoostingClassifier as XGBClassifier

# Paths
base_dir = Path('..')
processed_dir = base_dir / 'data' / 'processed'
splits_dir = base_dir / 'data' / 'splits'
models_dir = base_dir / 'models'
results_dir = base_dir / 'results'

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

# Load splits created in Phase 5
X_train = pd.read_csv(splits_dir / 'X_train_balanced.csv')
X_test = pd.read_csv(splits_dir / 'X_test.csv')
y_train = pd.read_csv(splits_dir / 'y_train_balanced.csv').squeeze()
y_test = pd.read_csv(splits_dir / 'y_test.csv').squeeze()

# Load scaled splits for linear models
X_train_scaled = pd.read_csv(splits_dir / 'X_train_scaled.csv')
X_test_scaled = pd.read_csv(splits_dir / 'X_test_scaled.csv')

# Load feature names if available
feature_names_path = processed_dir / 'feature_names.pkl'
if feature_names_path.exists():
    with open(feature_names_path, 'rb') as f:
        all_features = pickle.load(f)
else:
    all_features = X_train.columns.tolist()

# Define red flag features used for analysis
red_flag_features = [
    'has_wire_transfer',
    'has_upfront_payment',
    'has_unverified_company',
    'has_generic_location',
    'has_urgency_language',
    'has_too_good_promise',
    'has_data_entry_money',
    'has_missing_logo',
    'has_vague_salary',
    'has_poor_grammar',
    'red_flag_count',
    'has_multiple_red_flags',
    'fraud_risk_score',
]
red_flag_features = [f for f in red_flag_features if f in all_features]

print('Loaded data and configuration:')
print('  X_train:', X_train.shape)
print('  X_test :', X_test.shape)
print('  y_train:', y_train.shape)
print('  y_test :', y_test.shape)
print('  Scaled X_train:', X_train_scaled.shape)
print('  Scaled X_test :', X_test_scaled.shape)
print('  Feature count:', len(all_features))
print('  Red flag features found:', len(red_flag_features))


Loaded data and configuration:
  X_train: (26698, 371)
  X_test : (3507, 371)
  y_train: (26698,)
  y_test : (3507,)
  Scaled X_train: (26698, 371)
  Scaled X_test : (3507, 371)
  Feature count: 371
  Red flag features found: 13


In [2]:
# Train classification models and an XGBoost regressor

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

if xgb_available:
    xgb_clf = XGBClassifier(
        n_estimators=150,
        max_depth=7,
        learning_rate=0.1,
        scale_pos_weight=max(1, int((len(y_train) - y_train.sum()) / max(1, y_train.sum()))),
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
    )
else:
    xgb_clf = XGBClassifier(random_state=42)

xgb_clf.fit(X_train, y_train)

xgb_reg = None
if xgb_available:
    xgb_reg = XGBRegressor(
        n_estimators=150,
        max_depth=7,
        learning_rate=0.1,
        random_state=42,
    )
    xgb_reg.fit(X_train, y_train)

lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
)
lr_model.fit(X_train_scaled, y_train)

models = {
    'Random Forest': rf_model,
    'XGBoost Classifier': xgb_clf,
    'Logistic Regression': lr_model,
}
if xgb_reg is not None:
    models['XGBoost Regressor'] = xgb_reg

print('Trained models:')
for name in models:
    print(' -', name)


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:05:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Trained models:
 - Random Forest
 - XGBoost Classifier
 - Logistic Regression
 - XGBoost Regressor


In [3]:
# Make predictions with each trained model and evaluate performance

predictions = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        X_eval = X_test_scaled
    else:
        X_eval = X_test

    if name == 'XGBoost Regressor':
        preds_raw = model.predict(X_eval)
        probs = np.clip(preds_raw, 0.0, 1.0)
        preds = (probs >= 0.5).astype(int)
    else:
        probs = model.predict_proba(X_eval)[:, 1]
        preds = (probs >= 0.5).astype(int)

    predictions[name] = {
        'predictions': preds,
        'probabilities': probs,
    }

print('Generated predictions for models:')
for name in predictions:
    print(' -', name)


Generated predictions for models:
 - Random Forest
 - XGBoost Classifier
 - Logistic Regression
 - XGBoost Regressor


In [4]:
# Evaluate models, compare results, and save outputs

results = []
for name, result in predictions.items():
    y_pred = result['predictions']
    y_prob = result['probabilities']

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    print(f"\n{'='*60}")
    print(f'MODEL: {name}')
    print('Confusion Matrix:')
    print(f'  TN: {tn}  FP: {fp}  FN: {fn}  TP: {tp}')
    print('Metrics:')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  F1-score:  {f1:.4f}')
    print(f'  ROC AUC:   {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))

    results.append({
        'model': name,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': auc,
    })

results_df = pd.DataFrame(results)
results_df.to_csv(results_dir / 'model_comparison.csv', index=False)

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=False)
feature_importance.to_csv(results_dir / 'feature_importance.csv', index=False)

if red_flag_features:
    features_without_mining = [f for f in all_features if f not in red_flag_features]
    if features_without_mining:
        rf_no_mining = RandomForestClassifier(
            n_estimators=150,
            max_depth=15,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
        )
        rf_no_mining.fit(X_train[features_without_mining], y_train)
        y_no_mining = rf_no_mining.predict(X_test[features_without_mining])
        f1_no_mining = f1_score(y_test, y_no_mining)
        f1_with_mining = f1_score(y_test, predictions['Random Forest']['predictions'])
        improvement = f1_with_mining - f1_no_mining
        pct = (improvement / f1_no_mining * 100) if f1_no_mining > 0 else np.nan
        pd.DataFrame([
            {'metric': 'f1_with_mining', 'value': f1_with_mining},
            {'metric': 'f1_without_mining', 'value': f1_no_mining},
            {'metric': 'f1_improvement', 'value': improvement},
            {'metric': 'f1_improvement_pct', 'value': pct},
        ]).to_csv(results_dir / 'mining_impact.csv', index=False)

for name, model in models.items():
    model_path = models_dir / f"model_{name.replace(' ', '_').lower()}.pkl"
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)

predictions_df = pd.DataFrame({
    'true_fraudulent': y_test.values,
    'rf_prediction': predictions['Random Forest']['predictions'],
    'rf_probability': predictions['Random Forest']['probabilities'],
    'xgb_clf_prediction': predictions['XGBoost Classifier']['predictions'],
    'xgb_clf_probability': predictions['XGBoost Classifier']['probabilities'],
    'lr_prediction': predictions['Logistic Regression']['predictions'],
    'lr_probability': predictions['Logistic Regression']['probabilities'],
})
if 'XGBoost Regressor' in predictions:
    predictions_df['xgb_reg_prediction'] = predictions['XGBoost Regressor']['predictions']
    predictions_df['xgb_reg_probability'] = predictions['XGBoost Regressor']['probabilities']

predictions_df.to_csv(results_dir / 'model_predictions.csv', index=False)
print('Saved evaluation artifacts to results folder.')



MODEL: Random Forest
Confusion Matrix:
  TN: 3310  FP: 28  FN: 45  TP: 124
Metrics:
  Precision: 0.8158
  Recall:    0.7337
  F1-score:  0.7726
  ROC AUC:   0.9837
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99      3338
  Fraudulent       0.82      0.73      0.77       169

    accuracy                           0.98      3507
   macro avg       0.90      0.86      0.88      3507
weighted avg       0.98      0.98      0.98      3507


MODEL: XGBoost Classifier
Confusion Matrix:
  TN: 3323  FP: 15  FN: 41  TP: 128
Metrics:
  Precision: 0.8951
  Recall:    0.7574
  F1-score:  0.8205
  ROC AUC:   0.9906
              precision    recall  f1-score   support

  Legitimate       0.99      1.00      0.99      3338
  Fraudulent       0.90      0.76      0.82       169

    accuracy                           0.98      3507
   macro avg       0.94      0.88      0.91      3507
weighted avg       0.98      0.98      0.98      3507


MODEL: Log

In [7]:
from sklearn.ensemble import GradientBoostingClassifier

best_model_name = results_df.loc[results_df['f1_score'].idxmax(), 'model']
best_f1 = results_df['f1_score'].max()
print(f"Best model: {best_model_name} with F1 = {best_f1:.4f}")

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    random_state=42
)
gb_model.fit(X_train, y_train)

gb_preds = gb_model.predict(X_test)
gb_probs = gb_model.predict_proba(X_test)[:, 1]
gb_f1 = f1_score(y_test, gb_preds)
gb_auc = roc_auc_score(y_test, gb_probs)

print(f"\nGradient Boosting Results:")
print(f"  F1-Score: {gb_f1:.4f}")
print(f"  ROC AUC:  {gb_auc:.4f}")
print(classification_report(y_test, gb_preds, target_names=['Legitimate', 'Fraudulent']))

model_path = models_dir / 'model_gradient_boosting.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(gb_model, f)

print(f"Saved Gradient Boosting model to {model_path}")

Best model: XGBoost Classifier with F1 = 0.8205

Gradient Boosting Results:
  F1-Score: 0.7674
  ROC AUC:  0.9824
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99      3338
  Fraudulent       0.78      0.75      0.77       169

    accuracy                           0.98      3507
   macro avg       0.89      0.87      0.88      3507
weighted avg       0.98      0.98      0.98      3507

Saved Gradient Boosting model to ..\models\model_gradient_boosting.pkl
